In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from core.Log import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
from tqdm.notebook import tqdm
OUTER_FOLDS = 5; INNER_FOLDS = 3
from core.modelUtils import *
import logging
setup_loggers()


C:\Users\sulei\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [ ]:
#from sklearn.model_selection import train_test_split
#main_dataset = load_dataset_info(file="data/data_info.json")
#labels = [lbl['label'] for lbl in main_dataset]
#main_training, final_test = train_test_split(main_dataset,
#											 test_size=22,
#											 stratify=labels,
#											 random_state=67)
#print(len(final_test))
#
#for sample in main_dataset:
#	if sample in final_test: sample['pool'] = 'holdout'
#	else: sample['pool'] = 'main'

#save_dataset_info(main_dataset, file="data/data_info.json")


22


In [12]:
#create_folds_stats(OUTER_K=5, INNER_K=3)


Generating and saving fold indices...
OUTER FOLD 0 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 1 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 2 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 3 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samp

In [ ]:
def save_to_json(data, filename="training/INNER_FOLDS.json"):
	#print(f"Saving to {filename}.")
	os.makedirs(os.path.dirname(filename), exist_ok=True)
	with open(filename, 'w') as f:
		json.dump(data, f, indent=2)

def load_from_json(filename="training/INNER_FOLDS.json"):
	if not os.path.exists(filename):
		print(f"No data found in {filename}.")
		return []
	with open(filename, 'r') as f:
		results = json.load(f)
		print(f"Loaded {filename}.")
		return results




48


[{'ExpID': 1,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 2,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 2,
   'LR': 0.0005,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 3,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 3,
   'LR': 0.0004,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 4,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 4,
   'LR': 0.0003,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 5,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 1,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 6,
  'OUTER_FOLD': 1,
  'INNER_FOLD': 1,
  '

In [19]:
INNER_experiments = load_from_json("training/INNER_FOLDS.json")
len(INNER_experiments)

# Filter experiments for OUTER_FOLD 0
exp_list = [item for item in INNER_experiments if item['OUTER_FOLD'] == 0 and item['Model'] == "RESNET_18"]
print(f"parameter sets {len(exp_list)}")


Loaded training/INNER_FOLDS.json.
parameter sets 12


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import logging
from tqdm.notebook import tqdm
import torch.nn as nn
from core.benchmarks import *
setup_loggers()
def train_INNER_MLP(model, train_loader, val_loader, experiment):
	''' DONE. DONT CHANGE IT EVER..'''
	log = logging.getLogger('INNER_train')
	#log.info(f"		 ExpID; OUTER_FOLD; INNER_FOLD;	HP_Set;   Epoch;  TrainLoss;  ValLoss;  P;  LR")
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	hypers = experiment['hypers']
	#LR = hypers['LR']
	#WD = hypers['WD']
	#ExpID = experiment['ExpID']
	#P = hypers['P']
	epochs = hypers['Epochs']
	optimizer = optim.Adam(model.parameters(), lr= hypers['LR'], weight_decay=hypers['WD'])
	scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
	criterion = nn.BCEWithLogitsLoss()
	best_V_loss = float('inf')
	P_counter = 0
	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)
	pbar_epochs = tqdm(range(epochs), desc=f"	↳ Experiment {experiment['ExpID']} | Training model... ", position=experiment['ExpID'], leave=True)
	for epoch in pbar_epochs:
		model.train()
		running_loss = 0.0
		for batch in train_loader:
			#axi = batch["axial_image"].to(device)
			#cor = batch["coronal_image"].to(device)
			#sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)
			optimizer.zero_grad()
			outputs = model(met)
			T_loss = criterion(outputs, lbl)
			T_loss.backward()
			optimizer.step()
			running_loss += T_loss.item() * lbl.size(0)
		T_loss = running_loss / train_N
		model.eval()
		running_loss = 0.0
		with torch.no_grad():
			for batch in val_loader:
				#axi = batch["axial_image"].to(device)
				#cor = batch["coronal_image"].to(device)
				#sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)
				outputs = model(met)
				V_loss = criterion(outputs, lbl)
				running_loss += V_loss.item() * lbl.size(0)

		V_loss = running_loss / val_N
		scheduler.step(V_loss)
		if V_loss < best_V_loss:
			best_V_loss = V_loss
			P_counter = 0
		else:
			P_counter += 1
		log.info(f"	 MLP;	{experiment['ExpID']}; 	{experiment['OUTER_FOLD']}; 	{experiment['INNER_FOLD']}; 	{hypers['HPset']};  	{epoch}; 	{T_loss}; 	{V_loss};  	{P_counter}; 	{optimizer.param_groups[0]['lr']}")
		if P_counter >= hypers['P']: break
	return best_V_loss




def train_INNER_RESNET(train_loader, val_loader, experiment):
	''' DONE. DONT CHANGE IT EVER..'''
	log = logging.getLogger('INNER_train')
	#log.info(f"		 ExpID; OUTER_FOLD; INNER_FOLD;	HP_Set;   Epoch;  TrainLoss;  ValLoss;  P;  LR")
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	feature_extractor, classifier = create_adapted_resnet18(device)
	#model.to(device)
	hypers = experiment['hypers']
	epochs = hypers['Epochs']
	optimizer = optim.Adam(classifier.parameters(), lr= hypers['LR'], weight_decay=hypers['WD'])
	scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
	criterion = nn.BCEWithLogitsLoss()
	best_V_loss = float('inf')
	P_counter = 0
	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)
	pbar_epochs = tqdm(range(epochs), desc=f"	↳ RESNET Experiment {experiment['ExpID']} | Training model... ", position=experiment['ExpID'], leave=True)
	for epoch in pbar_epochs:

		classifier.train()
		feature_extractor.eval()

		running_loss = 0.0
		for batch in train_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)
			with torch.no_grad():
				axi_features = feature_extractor(axi)
				cor_features = feature_extractor(cor)
				sag_features = feature_extractor(sag)

			combined_input = torch.cat([axi_features, cor_features, sag_features, met], dim=1)

			optimizer.zero_grad()
			outputs = classifier(combined_input)
			T_loss = criterion(outputs, lbl)
			T_loss.backward()
			optimizer.step()
			running_loss += T_loss.item() * lbl.size(0)

		T_loss = running_loss / train_N

		classifier.eval()
		running_loss = 0.0
		with torch.no_grad():
			for batch in val_loader:
				axi = batch["axial_image"].to(device)
				cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				axi_features = feature_extractor(axi)
				cor_features = feature_extractor(cor)
				sag_features = feature_extractor(sag)

				combined_input = torch.cat([axi_features, cor_features, sag_features, met], dim=1)
				outputs = classifier(combined_input)
				V_loss = criterion(outputs, lbl)
				running_loss += V_loss.item() * lbl.size(0)

		V_loss = running_loss / val_N
		scheduler.step(V_loss)

		if V_loss < best_V_loss:
			best_V_loss = V_loss
			P_counter = 0
		else:
			P_counter += 1
		log.info(f"	 RESNET;	{experiment['ExpID']}; 	{experiment['OUTER_FOLD']}; 	{experiment['INNER_FOLD']}; 	{hypers['HPset']};  	{epoch}; 	{T_loss}; 	{V_loss};  	{P_counter}; 	{optimizer.param_groups[0]['lr']}")
		if P_counter >= hypers['P']: break
	return best_V_loss


In [ ]:
DL = DataLoaderFactory(load_dataset(pool="main"), get_fold_stats())
log = logging.getLogger('INNER_train')
#pbar_experiments = tqdm(exp_list, desc=f"OUTER FOLD 0 HP SEARCH", position=0, leave=True)
log.info(f"------------------------------  BENCHMARKS   --------------------------------------------")
INNER_experiments = load_from_json("training/INNER_FOLDS.json")
log = logging.getLogger('INNER_train')
for experiment in INNER_experiments:
	if experiment['trained'] == True: continue
	if experiment['OUTER_FOLD'] != 0: continue
	print(f"OUT : {experiment['OUTER_FOLD']}")
	if experiment['Model'] == "RESNET_18":
		hypers = experiment['hypers']
		#DR = hypers['DR']
		OUT = experiment['OUTER_FOLD']
		INN = experiment['INNER_FOLD']
		train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
		#model = MetadataMLP(hypers['DR'])
		#best_val_loss = train_INNER_MLP(model, train_loader, val_loader, experiment)
		best_val_loss = train_INNER_RESNET(train_loader, val_loader, experiment)
		experiment['best_val_loss'] = best_val_loss
		experiment['trained'] = True
	elif experiment['Model'] == "MLP":
		hypers = experiment['hypers']
		#DR = hypers['DR']
		OUT = experiment['OUTER_FOLD']
		INN = experiment['INNER_FOLD']
		train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
		model = MetadataMLP(hypers['DR'])
		best_val_loss = train_INNER_MLP(model, train_loader, val_loader, experiment)
		#best_val_loss = train_INNER_RESNET(train_loader, val_loader, experiment)
		experiment['best_val_loss'] = best_val_loss
		experiment['trained'] = True
	save_to_json(INNER_experiments, filename="training/INNER_experiments.json")



